# Getting started - PII Detector Model

Detect and anonymize PII in English text (names, emails, SSNs, credit cards, IBANs, IPs, URLs, and more) on a SageMaker endpoint.

This notebook deploys the Sigmodata SageMaker model package from AWS Marketplace, sends example texts, prints detected spans and the anonymized string, then deletes the endpoint.

**Before you run this:**
1. Subscribe on AWS Marketplace (**Continue to Subscribe** -> **Continue to Configuration**).
2. Copy the **Model Package ARN** from that page into the next cell. That ARN is AWS's copy of the package (unique per region), not the seller ARN.
3. Run from a SageMaker Notebook / Studio session (execution role auto-detected), or set `role` manually.

Product page: https://www.sigmodata.com/products/pii-detector-model  
Download this notebook: https://www.sigmodata.com/products/pii-detector-model/getting-started.ipynb

In [ ]:
# Product ARN from AWS Marketplace -> Continue to Configuration.
# That is AWS's copy of the model package (unique per region). Do not use a
# seller-account package ARN.
MODEL_PACKAGE_ARN = ""
assert MODEL_PACKAGE_ARN.startswith("arn:aws:sagemaker:"), (
    "Paste the Model Package ARN from Marketplace Continue to Configuration"
)


## 1. Deploy a real-time endpoint

Persistent endpoints bill per hour while running. For bulk files, skip to **Batch transform** - you only pay for job runtime. CPU only; `ml.m5.large` is the recommended instance.

In [ ]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.model import ModelPackage

session = sagemaker.Session()
role = get_execution_role()

model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)

predictor = model.deploy(initial_instance_count=1, instance_type="ml.m5.large")

## 2. Detect and anonymize example texts

**Request:** `{"texts": ["...", "..."]}`  
**Response:** `{"results": [{"text": "<anonymized>", "entities": [{entity_type, start, end, score, text}, ...]}, ...]}` - one entry per input, in order.

Optional `entities` (allowlist), `operators` (per-type replace/mask/redact/hash/encrypt), and `language` (default `en`). `input` is accepted as an alias for `texts`.

In [ ]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

texts = [
    "My name is Jamie Rivera, email test.user@example.com, phone 212-555-0100.",
    "SSN 234-56-7890, card number 4111 1111 1111 1111.",
]

result = predictor.predict({"texts": texts})
for original, row in zip(texts, result["results"]):
    types = ", ".join(e["entity_type"] for e in row["entities"])
    print(f"in:  {original}")
    print(f"out: {row['text']}")
    print(f"     {types}\n")

## 3. Per-entity operators

Mask credit cards, replace everything else. The key is never stored server-side; `hash` can take an optional caller-supplied salt for a stable join key.

In [ ]:
payload = {
    "texts": ["SSN 234-56-7890, card number 4111 1111 1111 1111."],
    "operators": {
        "DEFAULT": {"type": "replace", "new_value": "<REDACTED>"},
        "CREDIT_CARD": {"type": "mask", "masking_char": "*", "chars_to_mask": 12, "from_end": False},
    },
}
print(predictor.predict(payload))

## 4. Clean up the endpoint

Real-time endpoints keep billing until deleted. Always run this when finished.

In [ ]:
predictor.delete_endpoint()

## Batch transform (optional)

Cheaper for bulk jobs: CSV in S3 (one text per line, no header) -> `text,entities_json` CSV out.

In [ ]:
import boto3

bucket = session.default_bucket()
input_key = "sigmodata-example/pii-detector-model/texts.csv"

sample_texts = (
    "My name is Jamie Rivera, email test.user@example.com.\n"
    "SSN 234-56-7890, card number 4111 1111 1111 1111.\n"
)
boto3.client("s3").put_object(Bucket=bucket, Key=input_key, Body=sample_texts.encode())

input_s3_uri = f"s3://{bucket}/{input_key}"
output_s3_uri = f"s3://{bucket}/sigmodata-example/pii-detector-model/output/"
print("Input:", input_s3_uri)
print("Output:", output_s3_uri)

In [ ]:
transformer = model.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=output_s3_uri,
    strategy="MultiRecord",
    max_payload=6,
)

transformer.transform(
    data=input_s3_uri,
    content_type="text/csv",
    split_type="Line",
    wait=True,
)

In [ ]:
output_key = f"{input_key.rsplit('/', 1)[-1]}.out"
obj = boto3.client("s3").get_object(
    Bucket=bucket, Key=f"sigmodata-example/pii-detector-model/output/{output_key}"
)
print(obj["Body"].read().decode())

## Next steps

- Product page: https://www.sigmodata.com/products/pii-detector-model
- Support: support@sigmodata.com